# Continual Learning in Token Space: A Practical Guide

MemoRizz learns here by changing the tokens supplied to an agent at inference time, not
by updating model weights.

The MemoRizz harness performs four steps:

- verify and record tool runs;
- group structurally equivalent runs;
- distill their shared procedure into a reviewed skill; and
- retrieve that skill for semantically similar requests.


```mermaid
flowchart LR
    A[Agent tool run] --> W[(Workflow evidence)]
    W --> C[Canonicalize repeated paths]
    C --> G{Evidence gates pass?}
    G -- yes --> S[Distill and review skill]
    G -- no --> W
    S --> R[Retrieve for a matching request]
    R --> P[Add skill tokens to context]
    P --> A
```

### Definitions

| Term | Meaning in this notebook |
|---|---|
| **Workflow (trajectory)** | One observed agent run: the user request, ordered tool calls, argument shapes, results or errors, and a business-graded outcome. It is evidence of what happened once, not an instruction by itself. |
| **Canonicalization** | Normalize a concrete workflow into a structural signature so runs with different order IDs or result values—but the same ordered tools, argument shapes, and success/error pattern—can be grouped as one procedure. |
| **Distillation** | Synthesize several verified workflows from one canonical group into a compact, generalized skill containing applicability, preconditions, ordered actions, tools, and failure handling. |
| **Skill** | A compact, reusable instruction artifact distilled from several comparable successful workflows. It records when the procedure applies, its preconditions, ordered steps, tools, and failure handling. |
| **Continual learning** | The post-deployment loop that keeps collecting outcomes and updates which reusable instructions are stored, promoted, retrieved, monitored, or demoted. Here it changes memory and context tokens; it does **not** fine-tune the LLM. |

  ### Use case: Compile repeated workflows into reusable skills

  This guide demonstrates how an agent can turn repeated, verified executions into reusable procedural knowledge without fine-tuning the underlying
  model.

  The agent initially performs a task using detailed coaching instructions and application tools. 
  
  MemoRizz records each execution as a workflow containing the request, ordered tool calls, argument shapes, results, errors, and business-graded outcome. It then canonicalizes successful workflows so executions with different input values—but the same underlying procedure—are grouped together.

  After a workflow group satisfies execution-count, success-rate, and query-diversity requirements, MemoRizz distills the shared procedure into a
  compact skill.
  
  The skill describes when the procedure applies, its preconditions, ordered actions, required tools, and stop or failure conditions.
  
  A reviewed and activated skill can then be retrieved for semantically similar requests.

  At inference time, each skill agent receives the compact skill rather than the raw workflows from which it was produced.
  
  **The original workflows remain in persistent storage for auditing, monitoring, and future skill improvement, but they are not replayed in either skill agent's prompt.**

  The refund procedure in this notebook is one concrete example: check current order state, perform an eligible refund, and send a receipt. 
  
  The same pattern applies to other repeatable tool-based processes such as ticket triage, customer onboarding, claims processing, incident response, compliance checks, and data operations.

  The final evaluation compares three ways of supplying the same procedural evidence:

  1. A baseline that replays complete successful workflows.
  2. A continual-learning agent that receives the distilled skill as user context.
  3. A continual-learning agent that receives the same reviewed skill as a developer instruction.

  All three arms use the same model, tools, production instruction, held-out requests, and independently ingested database state. 
  This isolates both the workflow-to-skill compression effect and the effect of giving the same reviewed procedure higher instruction authority.


### Why move from workflows to skills in token space?

| Raw workflow history | Retrieved skill |
|---|---|
| Contains request-specific IDs, intermediate results, and repeated text. | Compresses the invariant procedure and omits incidental values. |
| Becomes expensive and distracting if many complete traces are placed in every prompt. | Adds one bounded instruction only when semantic retrieval says it is relevant. |
| Describes past behavior but may not clearly tell the model what to do next. | States preconditions, ordered actions, and stop conditions as executable guidance. |
| Is difficult to govern as an instruction. | Can be reviewed in shadow state, activated, attributed, monitored, versioned, and demoted without changing model weights. |


This is an inference-time analogue of **progressive disclosure**. 

OpenAI's
[skill documentation](https://learn.chatgpt.com/docs/build-skills) describes loading small
skill metadata first and full instructions only after selection, specifically to preserve
context. OpenAI's engineering post
[From model to agent](https://openai.com/index/equip-responses-api-computer-environment/)
similarly describes skills as reusable workflow logic loaded into model context and
compaction as a way to retain high-value state in a token-efficient form.

Role placement still matters. OpenAI's
[2024 instruction-hierarchy research](https://openai.com/index/the-instruction-hierarchy/)
and its
[2026 follow-up](https://openai.com/index/instruction-hierarchy-challenge/) explain that
models should prefer more trusted instruction sources when messages conflict. A file named
`SKILL.md` is not automatically privileged; message role determines authority. MemoRizz now
stores `skill_injection_role` on each learned skill. The safe default, `user`, renders the
skill in volatile user context. The opt-in `developer` setting is allowed only with shadow
review and activation, then sends the reviewed skill as a separate developer message below
system policy and above the user request. This follows OpenAI's
[message-role guidance](https://developers.openai.com/api/docs/guides/text#message-roles-and-instruction-following).
Anthropic has no portable developer message role, so MemoRizz maps the same reviewed
application instruction into Anthropic's [top-level `system` parameter](https://platform.claude.com/docs/en/api/messages/create). In every mode the
agent must verify skill preconditions and current tool results before acting.

Selective context can improve accuracy, but retrieval and extra prompt tokens can also add
cost and latency. The final three-arm evaluation therefore reports answer accuracy, exact
workflow accuracy, combined task accuracy, prompt/completion tokens, LLM calls, provider
inference latency, and end-to-end latency rather than assuming a free improvement.

## 1. Set Up the Runtime

Use a Jupyter kernel that has the released MemoRizz package and pandas installed. This guide
imports MemoRizz directly from that environment; it does not discover a repository, modify
`sys.path`, or import from a local `src` tree. Install or update the runtime before opening
the notebook:

```bash
python -m pip install --upgrade "memorizz[oracle,ollama]" "pandas>=2.0"
```

The first Python cell prints the imported package version and location. Confirm that the
location belongs to the selected environment's `site-packages` directory.

The notebook expects these environment variables, usually loaded from an ignored `.env`:

- `ORACLE_USER`, `ORACLE_PASSWORD`, and `ORACLE_DSN`
- `OPENAI_API_KEY` for 256-dimensional embeddings
- optionally `ORACLE_SCHEMA`

For a new schema, run this once in a terminal before opening the notebook:

```bash
export ORACLE_EMBEDDING_DIM=256
memorizz setup-oracle
```

OpenAI is the default reasoning provider.

To use Ollama for agent reasoning, set
`MEMORIZZ_GUIDE_LLM_PROVIDER=ollama`; this guide still uses OpenAI embeddings, so
`OPENAI_API_KEY` remains required. 

It makes real model calls and writes workflow, skill,
tool, and isolated demo-order rows to Oracle. No credential value is printed.

In [ ]:
! pip install --upgrade "memorizz[oracle,ollama]" "pandas>=2.0"

In [ ]:
import os
import uuid
from importlib.metadata import version
from pathlib import Path

import memorizz
from dotenv import find_dotenv, load_dotenv


env_file = find_dotenv(usecwd=True)
if env_file:
    load_dotenv(env_file, override=False)

os.environ.setdefault("MEMORIZZ_GUIDE_LLM_PROVIDER", "openai")
os.environ.setdefault("OPENAI_MODEL", "gpt-5.6")
os.environ.setdefault("OPENAI_REASONING_EFFORT", "none")
os.environ.setdefault("OLLAMA_HOST", "http://127.0.0.1:11434")
os.environ.setdefault("OLLAMA_MODEL", "kimi-k2.7-code:cloud")
os.environ.setdefault("OLLAMA_THINK", "false")
os.environ.setdefault("ORACLE_SCHEMA", os.environ.get("ORACLE_USER", ""))
os.environ.setdefault("MEMORIZZ_DISABLE_CONVERSATION_EMBEDDINGS", "1")

required_keys = ("ORACLE_USER", "ORACLE_PASSWORD", "ORACLE_DSN", "OPENAI_API_KEY")
missing_keys = [key for key in required_keys if not os.environ.get(key)]
if missing_keys:
    raise RuntimeError(f"Missing environment variables: {missing_keys}")


def required_env(name: str) -> str:
    return os.environ[name]


def env_flag(name: str, default: bool = False) -> bool:
    value = os.getenv(name)
    if value is None:
        return default
    return value.strip().lower() in {"1", "true", "yes", "on"}


LLM_PROVIDER = os.environ["MEMORIZZ_GUIDE_LLM_PROVIDER"].strip().lower()
if LLM_PROVIDER not in {"openai", "ollama"}:
    raise RuntimeError("MEMORIZZ_GUIDE_LLM_PROVIDER must be 'openai' or 'ollama'.")

OPENAI_MODEL = os.environ["OPENAI_MODEL"]
OPENAI_REASONING_EFFORT = os.environ["OPENAI_REASONING_EFFORT"]
OLLAMA_MODEL = os.environ["OLLAMA_MODEL"]
OLLAMA_HOST = os.environ["OLLAMA_HOST"]
RUN_ID = os.getenv("MEMORIZZ_GUIDE_RUN_ID", uuid.uuid4().hex[:10])
BASELINE_AGENT_ID = f"continual-learning-baseline-{RUN_ID}"
USER_SKILL_AGENT_ID = f"continual-learning-user-skill-{RUN_ID}"
DEVELOPER_SKILL_AGENT_ID = f"continual-learning-developer-skill-{RUN_ID}"
DEMO_USER_ID = f"guide-user-{RUN_ID}"

print(
    {
        "memorizz_version": version("memorizz"),
        "memorizz_source": str(
            Path(memorizz.__file__).resolve()
            if memorizz.__file__
            else list(memorizz.__path__)
        ),
        "environment_file": env_file or "<not found>",
        "run_id": RUN_ID,
        "llm_provider": LLM_PROVIDER,
    }
)

## 2. Connect Oracle, Embeddings, and the LLM

The Oracle schema and embedding provider must use the same vector dimension.

This guide uses `text-embedding-3-small` with 256 values and validates the schema before
continuing. 

The embedding vectors support semantic retrieval of tools, workflows, and
skills; ordinary order lookup later uses relational SQL rather than vector search.

Oracle VECTOR column dimensions are fixed in table DDL. Changing the embedding dimension
therefore requires a new schema or a deliberate migration and re-embedding process.

In [ ]:
import os

from memorizz.embeddings import get_embedding
from memorizz.memory_provider.oracle import OracleConfig, OracleProvider


oracle_user = required_env("ORACLE_USER")

provider = OracleProvider(
    OracleConfig(
        user=oracle_user,
        password=required_env("ORACLE_PASSWORD"),
        dsn=required_env("ORACLE_DSN"),
        schema=os.getenv("ORACLE_SCHEMA") or oracle_user,
        lazy_vector_indexes=True,
        in_database_embedding=False,
        embedding_provider="openai",
        embedding_config={
            "model": "text-embedding-3-small",
            "dimensions": 256,
        },
    )
)

### Validate the vector contract

`validate_vector_schema_dimensions(256)` inspects every relevant Oracle VECTOR column and
fails before the experiment if any column was created with a different width. This avoids
a delayed `ORA-51803` during a later workflow or skill write. `get_embedding(...)` then
makes one real embedding request; checking its length proves that the configured provider
actually emits the same 256-dimensional vectors expected by the schema. The printed schema
name and dimension are safe diagnostics and contain no credentials.

In [ ]:
provider.validate_vector_schema_dimensions(256)
probe_vector = get_embedding("refund a completed order and send a receipt")
print({"schema": provider.config.schema, "embedding_dimensions": len(probe_vector)})

In [ ]:
import os
from memorizz.llms.ollama import OllamaLLM
from memorizz.llms.openai import OpenAI


def make_llm():
    if LLM_PROVIDER == "openai":
        return OpenAI(
            model=OPENAI_MODEL,
            reasoning_effort=OPENAI_REASONING_EFFORT,
        )
    return OllamaLLM(
        model=OLLAMA_MODEL,
        host=OLLAMA_HOST,
        temperature=0,
        seed=7,
        num_predict=int(os.getenv("OLLAMA_NUM_PREDICT", "2048")),
        context_window_tokens=int(os.getenv("OLLAMA_CONTEXT_WINDOW", "32768")),
        timeout=600,
        think=env_flag("OLLAMA_THINK", False),
    )


### Experimental agents and what each arm isolates

The next cell creates three independent LLM provider instances, which are attached to three
`MemAgent` instances later in this section. All three use the same provider, model, model
settings, tools, seed requests, outcome rubric, and final production instruction. Separate
provider instances do **not** make them different models; they prevent mutable provider state
and per-call usage measurements from leaking between experimental arms.

| Agent / evaluation arm | Seed phase | Held-out procedural context | Skill authority | Raw workflows in the prompt? | Question this arm answers |
|---|---|---|---|---|---|
| **Baseline — raw workflow replay** | Receives the coached SOP and writes its own verified workflow records. Its Skillbox remains empty. | Receives the generic production instruction plus the successful source workflows as uncompressed, per-request context. | None | **Yes**—only this arm receives raw workflow records. | How well does replaying detailed demonstrations work before they are compiled into a skill? |
| **Skill — user authority** | Receives the same coached SOP and independently produces comparable workflow evidence. | Receives the generic production instruction plus one retrieved, reviewed, ACTIVE skill; source workflows are withheld. | The skill is rendered in the final `user` message's MemoRizz context block. | No | Does a compact skill preserve the procedure while reducing prompt tokens, and how strong is it when supplied as user-authority context? |
| **Skill — developer authority** | Receives the same coached SOP and independently produces comparable workflow evidence. | Receives the same generic production instruction and the **same reviewed skill content** as the user-skill arm; only placement changes. | The skill is emitted as a `developer` message for OpenAI, or mapped into Anthropic's top-level system instruction channel. | No | Does reviewed application-authority placement improve adherence relative to the identical user-authority skill? |

The arm names describe their eventual held-out treatment. During seed collection, no learned
skill is active yet: all three agents are deliberately coached to create comparable evidence.
After promotion and review, every arm switches to the same generic production instruction.
The controlled differences are then only **procedural representation** (raw workflows versus a
distilled skill) and, between the two skill arms, **instruction authority** (user versus
developer). This is therefore not a comparison of three model qualities.

In [ ]:
baseline_llm = make_llm()
user_skill_llm = make_llm()
developer_skill_llm = make_llm()
print(user_skill_llm.get_config())

## 3. Define the Refund Procedure

In a production design,
Oracle should be the transactional system of record (or a queryable replica populated by
CDC, events, batch loads, or an API ingestion job), while narrow tools retrieve and mutate
only the rows needed for the current request.

This guide creates a persistent `MEMORIZZ_GUIDE_ORDERS` table and an explicit ingestion
function. 

Each agent run receives its own snapshot, keyed by `experiment_id` and `run_id`,
so the three-arm benchmark is isolated. 

A production table would normally use tenant and order
keys instead; the extra run key is experimental scaffolding, not the recommended business
schema.

The tools enforce business state with Oracle transactions, while the agent decides which
tool to call next.

A valid refund run must:

1. look up the order;
2. refund it only when its status is `completed`; and
3. send a receipt after the refund succeeds.

The third step is the organization-specific rule that the agent will learn from examples.
The database still enforces the prerequisites, so a mistaken tool choice cannot bypass the
lookup, eligibility, or receipt state transitions.

### Oracle table, ingestion, and retrieval strategy

The next cell creates the shared demo table idempotently and defines
`ingest_order_snapshot`. The small `ORDER_SOURCE` mapping represents an upstream feed; it
is **not** queried by the agent tools. Ingestion copies one source record into Oracle for a
specific run, and every later tool reads the Oracle row.

| Retrieval need | Appropriate strategy | Why |
|---|---|---|
| Known `order_id` | Primary-key or unique-index lookup | Exact, fast, deterministic, and easy to authorize. This is what `lookup_order` uses. |
| Account/order filters | Parameterized SQL predicates, pagination, and indexes | Preserves relational semantics and avoids placing entire tables in the prompt. |
| Customer, payment, or shipment details | Relational joins or a governed view | Keeps referential integrity and access policy in the database. |
| Fuzzy search over notes or product descriptions | Oracle Text, VECTOR search, or hybrid lexical/vector retrieval | Useful when the user does not provide an exact key and meaning matters more than equality. |
| Current order state change | Transactional `UPDATE` with eligibility predicates | Makes the business precondition atomic and prevents a stale read-then-write race. |


```mermaid
flowchart LR
    U[Upstream order feed] --> I[Ingestion / CDC]
    I --> O[(Oracle orders table)]
    Q[User request] --> A[MemAgent]
    A --> L[lookup_order: exact SQL]
    L --> O
    A --> F[issue_refund: conditional UPDATE]
    F --> O
    A --> R[send_refund_receipt: conditional UPDATE]
    R --> O
```

For compactness the notebook uses the provider's pooled connection internally. A production
application should put the same parameterized statements behind a dedicated repository or
service boundary with tenant filters, least-privilege credentials, audit logging, retries,
and idempotency keys.

### Creating the ORDER TABLE

In [ ]:
import re
from memorizz import get_tool_context


schema_identifier = str(provider.config.schema or oracle_user).strip().upper()
if not re.fullmatch(r"[A-Z][A-Z0-9_$#]{0,127}", schema_identifier):
    raise RuntimeError(f"Unsafe Oracle schema identifier: {schema_identifier!r}")

ORDER_TABLE = f"{schema_identifier}.MEMORIZZ_GUIDE_ORDERS"
ORDER_SOURCE = {
    **{
        f"R-{number}": {
            "order_id": f"R-{number}",
            "status": "completed",
            "amount": 49.0,
        }
        for number in range(1001, 1017)
    },
    "R-1098": {"order_id": "R-1098", "status": "processing", "amount": 79.0},
    "R-1099": {"order_id": "R-1099", "status": "pending", "amount": 29.0},
}

create_orders_sql = f"""
CREATE TABLE {ORDER_TABLE} (
    experiment_id VARCHAR2(64) NOT NULL,
    run_id VARCHAR2(128) NOT NULL,
    order_id VARCHAR2(32) NOT NULL,
    status VARCHAR2(20) NOT NULL,
    amount NUMBER(12, 2) NOT NULL,
    lookup_count NUMBER DEFAULT 0 NOT NULL,
    receipt_sent NUMBER(1) DEFAULT 0 NOT NULL,
    updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP NOT NULL,
    PRIMARY KEY (experiment_id, run_id, order_id),
    CHECK (receipt_sent IN (0, 1))
)
"""

with provider._get_connection() as connection:
    cursor = connection.cursor()
    try:
        cursor.execute(create_orders_sql)
        connection.commit()
        table_state = "created"
    except Exception as exc:
        error_code = getattr(exc.args[0], "code", None) if exc.args else None
        if error_code != 955 and "ORA-00955" not in str(exc):
            raise
        connection.rollback()
        table_state = "already existed"
    finally:
        cursor.close()


In [ ]:
def ingest_order_snapshot(run_id: str, order_id: str) -> dict:
    """Replace one run-isolated Oracle row from the simulated source feed."""
    source = ORDER_SOURCE.get(order_id)
    with provider._get_connection() as connection:
        cursor = connection.cursor()
        try:
            cursor.execute(
                f"""
                DELETE FROM {ORDER_TABLE}
                WHERE experiment_id = :experiment_id
                  AND run_id = :run_id
                  AND order_id = :order_id
                """,
                {"experiment_id": RUN_ID, "run_id": run_id, "order_id": order_id},
            )
            if source is not None:
                cursor.execute(
                    f"""
                    INSERT INTO {ORDER_TABLE} (
                        experiment_id, run_id, order_id, status, amount,
                        lookup_count, receipt_sent
                    ) VALUES (
                        :experiment_id, :run_id, :order_id, :status, :amount, 0, 0
                    )
                    """,
                    {
                        "experiment_id": RUN_ID,
                        "run_id": run_id,
                        "order_id": order_id,
                        "status": source["status"],
                        "amount": source["amount"],
                    },
                )
            connection.commit()
        finally:
            cursor.close()
    return {"run_id": run_id, "order_id": order_id, "ingested": source is not None}


print({"order_table": ORDER_TABLE, "state": table_state})


In [ ]:
def _tool_run_id() -> str:
    context = get_tool_context() or {}
    run_id = str(context.get("run_id") or "").strip()
    if not run_id:
        raise RuntimeError("This demo tool requires tool_context['run_id'].")
    return run_id


In [ ]:
def order_snapshot(run_id: str, order_id: str):
    """Read the database state used by the independent experiment grader."""
    with provider._get_connection() as connection:
        cursor = connection.cursor()
        try:
            cursor.execute(
                f"""
                SELECT status, amount, lookup_count, receipt_sent
                FROM {ORDER_TABLE}
                WHERE experiment_id = :experiment_id
                  AND run_id = :run_id
                  AND order_id = :order_id
                """,
                {"experiment_id": RUN_ID, "run_id": run_id, "order_id": order_id},
            )
            row = cursor.fetchone()
        finally:
            cursor.close()
    if row is None:
        return None
    return {
        "order_id": order_id,
        "status": str(row[0]),
        "amount": float(row[1]),
        "lookup_count": int(row[2]),
        "receipt_sent": bool(row[3]),
    }


In [ ]:
def lookup_order(order_id: str) -> dict:
    """Fetch one exact order key from Oracle and record that it was checked."""
    run_id = _tool_run_id()
    with provider._get_connection() as connection:
        cursor = connection.cursor()
        try:
            cursor.execute(
                f"""
                SELECT status, amount
                FROM {ORDER_TABLE}
                WHERE experiment_id = :experiment_id
                  AND run_id = :run_id
                  AND order_id = :order_id
                """,
                {"experiment_id": RUN_ID, "run_id": run_id, "order_id": order_id},
            )
            row = cursor.fetchone()
            if row is None:
                return {"ok": False, "order_id": order_id, "error": "order_not_found"}
            cursor.execute(
                f"""
                UPDATE {ORDER_TABLE}
                SET lookup_count = lookup_count + 1,
                    updated_at = CURRENT_TIMESTAMP
                WHERE experiment_id = :experiment_id
                  AND run_id = :run_id
                  AND order_id = :order_id
                """,
                {"experiment_id": RUN_ID, "run_id": run_id, "order_id": order_id},
            )
            connection.commit()
        finally:
            cursor.close()
    return {
        "ok": True,
        "order_id": order_id,
        "status": str(row[0]),
        "amount": float(row[1]),
    }



In [ ]:
def issue_refund(order_id: str) -> dict:
    """Atomically refund a checked order only while its status is completed."""
    run_id = _tool_run_id()
    params = {"experiment_id": RUN_ID, "run_id": run_id, "order_id": order_id}
    with provider._get_connection() as connection:
        cursor = connection.cursor()
        try:
            cursor.execute(
                f"""
                UPDATE {ORDER_TABLE}
                SET status = 'refunded', updated_at = CURRENT_TIMESTAMP
                WHERE experiment_id = :experiment_id
                  AND run_id = :run_id
                  AND order_id = :order_id
                  AND status = 'completed'
                  AND lookup_count > 0
                """,
                params,
            )
            accepted = cursor.rowcount == 1
            if accepted:
                reason = "refunded"
            else:
                cursor.execute(
                    f"""
                    SELECT status, lookup_count
                    FROM {ORDER_TABLE}
                    WHERE experiment_id = :experiment_id
                      AND run_id = :run_id
                      AND order_id = :order_id
                    """,
                    params,
                )
                row = cursor.fetchone()
                reason = (
                    "order_not_found"
                    if row is None
                    else "lookup_required"
                    if int(row[1]) == 0
                    else f"status_{str(row[0]).lower()}"
                )
            connection.commit()
        finally:
            cursor.close()
    return {"ok": accepted, "order_id": order_id, "reason": reason}



In [ ]:
def send_refund_receipt(order_id: str) -> dict:
    """Mark the receipt sent only after Oracle records a successful refund."""
    run_id = _tool_run_id()
    params = {"experiment_id": RUN_ID, "run_id": run_id, "order_id": order_id}
    with provider._get_connection() as connection:
        cursor = connection.cursor()
        try:
            cursor.execute(
                f"""
                UPDATE {ORDER_TABLE}
                SET receipt_sent = 1, updated_at = CURRENT_TIMESTAMP
                WHERE experiment_id = :experiment_id
                  AND run_id = :run_id
                  AND order_id = :order_id
                  AND status = 'refunded'
                  AND receipt_sent = 0
                """,
                params,
            )
            accepted = cursor.rowcount == 1
            if accepted:
                reason = "receipt_sent"
            else:
                cursor.execute(
                    f"""
                    SELECT status, receipt_sent
                    FROM {ORDER_TABLE}
                    WHERE experiment_id = :experiment_id
                      AND run_id = :run_id
                      AND order_id = :order_id
                    """,
                    params,
                )
                row = cursor.fetchone()
                already_sent = bool(row and row[1])
                accepted = already_sent
                reason = (
                    "receipt_already_sent"
                    if already_sent
                    else "order_not_found"
                    if row is None
                    else "refund_required"
                )
            connection.commit()
        finally:
            cursor.close()
    return {"ok": accepted, "order_id": order_id, "reason": reason}

### Instruction design and the experimental contrast

| Instruction or constant | What it tells the agent | Why it exists |
|---|---|---|
| Coaching: `Call lookup_order` | Retrieve authoritative state before acting. | Creates verified demonstrations and prevents decisions from stale prompt text. |
| Coaching: refund only when `completed` | Gate the side effect on the returned status. | Encodes the business eligibility rule and a safe stop condition. |
| Coaching: send a receipt after refund success | Perform the organization-specific follow-up only after the refund tool confirms success. | This is the knowledge the later generic prompt omits and the learned skill should retain. |
| Coaching: stop for missing or ineligible orders | Do not attempt either downstream action. | Supplies negative boundaries and prevents harmful over-application. |
| Production: operate support with available tools | Gives all held-out arms the same role and broad objective. | Keeps the evaluation prompt realistic without leaking the hidden receipt SOP. |
| Production: check current data and trust tool eligibility | Preserves general safety requirements. | A skill may add a procedure, but it must not override current database facts. |
| `EXPECTED_REFUND_PATH` | Defines the exact three-tool success trajectory. | Gives workflow outcome grading and the held-out evaluator a deterministic target independent of final prose. |

The coaching prompt intentionally creates learning headroom; this is a test of retaining a
taught SOP, not discovering policy from first principles. All arms receive the same coached
runs. After promotion, all switch to the same production instruction; the two skill arms
receive the same active skill at different instruction roles.

In [ ]:
from memorizz import MemAgent
from memorizz.enums.memory_type import MemoryType
from memorizz.long_term.procedural.workflow import canonical_signature


COACHING_INSTRUCTION = """
Follow this refund procedure exactly:
1. Call lookup_order.
2. If the order is completed, call issue_refund.
3. If the refund succeeds, call send_refund_receipt.
Stop when an order is missing or ineligible.
""".strip()

PRODUCTION_INSTRUCTION = """
You operate order support. Complete the user's request with the available tools.
Check current data before making a change, and trust each tool's eligibility result.
""".strip()

EXPECTED_REFUND_PATH = [
    "lookup_order",
    "issue_refund",
    "send_refund_receipt",
]

### Promotion and retrieval configuration

| Setting | Meaning | Effect and trade-off |
|---|---|---|
| `min_executions=3` | Require three runs in one canonical trajectory class. | A low value makes the tutorial fast but increases overfitting risk; production generally needs more evidence. |
| `min_success_rate=0.80` | At least 80% of classed runs must have a successful business outcome. | Raising it improves precision but delays learning; lowering it can promote unreliable procedures. With only three accepted examples here, every collected seed must pass. |
| `min_distinct_queries=3` | Require three normalized request wordings. | Reduces promotion from repeated duplicates and encourages paraphrase coverage. |
| `distill_sample_size=3` | Give the distiller at most three successful examples. | More samples can improve coverage but increase distillation tokens, latency, and exposure to noisy traces. |
| `include_failure_samples=0` | Do not add failed traces to this small distillation prompt. | Keeps the demonstration deterministic, but loses failure-boundary evidence. Production skills should usually include curated failures when available. |
| `require_shadow=True` | Store a promoted candidate as non-retrievable `shadow` first. | Adds a review step and latency to activation, but prevents unreviewed instructions from affecting users. |
| `skill_injection_role` | Choose `user` or `developer` authority for newly promoted skills. | `user` is the backward-compatible default. `developer` is rejected unless shadow review is enabled. |
| `retrieval_min_similarity=0.35` | Accept only query/skill cosine matches at or above 0.35. | Lower values improve recall but risk negative transfer; higher values improve precision but can miss paraphrases. Tune on labeled retrieval data. |
| `max_skills_in_context=1` | Inject at most one learned skill per request. | Bounds prompt tokens and instruction conflicts; complex tasks may need a larger, carefully ranked set. |
| `promotion_every_n_runs=0` | Disable automatic periodic promotion. | This notebook calls `promote_class` explicitly so learning and evaluation phases cannot blur together. |

These values are tutorial controls, not universal defaults. The main quality levers are the
business-outcome grader, representative evidence, retrieval precision/recall, shadow review,
and post-activation monitoring—not simply lowering thresholds until a skill appears.

In [ ]:
PROMOTION_CONFIG = {
    "min_executions": 3,
    "min_success_rate": 0.80,
    "min_distinct_queries": 3,
    "distill_sample_size": 3,
    "include_failure_samples": 0,
    "require_shadow": True,
    "skill_injection_role": "user",
    "retrieval_min_similarity": 0.35,
    "max_skills_in_context": 1,
    "promotion_every_n_runs": 0,
}

DEVELOPER_PROMOTION_CONFIG = {
    **PROMOTION_CONFIG,
    "skill_injection_role": "developer",
}

### Grade workflows and construct comparable agents

`workflow_succeeded` is the application-level outcome callback. 

It canonicalizes the recorded
steps and derives the correct path from the authoritative lookup result: completed orders need
all three successful tools, while missing or ineligible orders must stop after lookup. MemoRizz
applies this label before promotion and monitoring see the workflow, so a run that merely
completed without raising an exception cannot count as business success—and a correct guardrail
stop is not mislabeled as a failure.

| `make_agent` argument or option | Purpose |
|---|---|
| `agent_id` | Isolates every arm's Oracle rows and later cleanup. |
| `model` / `llm_config` | Supplies the live provider instance and its serializable configuration. Separate instances make per-arm token and latency metering independent. |
| `instruction` | Starts all three arms with the coached SOP; all are switched to the same production prompt after promotion. |
| `WORKFLOW_MEMORY` | Persists each tool trajectory and its outcome for grouping and audit. |
| `SKILLBOX` and `continual_learning=True` | Gives all arms identical continual-learning scaffolding and static system instructions. Only the two treatment arms receive the reviewed skill. |
| `skill_injection_role` | Selects the persisted authority for promoted skills. Library validation requires shadow review before developer authority can be configured. |
| `max_steps=8` | Bounds the agent loop so a confused model cannot call tools indefinitely. |
| `semantic_cache=False` | Prevents a cached answer from replacing a fresh paired trial. |
| `continual_learning_config` | Applies the evidence and retrieval controls described above. |
| `workflow_outcome_evaluator` | Replaces exception-level success with the domain-specific path/result rubric. |
| `automations_enabled=False` | Removes unrelated background tools from the controlled tool surface. |
| Remove default tools, then `save()` | Leaves exactly the three refund tools and persists the parent agent row before Oracle receives agent-scoped Toolbox or Skillbox rows. |

#### `workflow_outcome_evaluator`: the learning-evidence gate

It is useful to call the evaluator a **gate**, with one important qualification: it is a
**post-run gate on learning evidence**, not a pre-execution authorization gate. The tool loop
and any side effects have already happened when MemoRizz invokes it. Business preconditions
must still be enforced before or inside the tool itself—as `issue_refund` does with a conditional
Oracle `UPDATE`.

```text
tool loop finishes
      ↓
captured Workflow (query, ordered tool calls, arguments, results, execution outcome)
      ↓
workflow_outcome_evaluator(workflow)
      ↓
persisted business outcome ──→ promotion evidence / active-skill monitoring / shadow metrics
```

Without an application evaluator, MemoRizz can only infer that a run succeeded because the
Python tool loop completed without raising. That is transport-level success, not necessarily
business success: a refund tool can return `{"ok": false, "reason": "lookup_required"}`
normally. Treating that run as successful would teach the continual-learning system from a
failed procedure.

| Question | MemoRizz behavior |
|---|---|
| **When is it called?** | After a tool-calling workflow is complete, but before the workflow is stored and before continual-learning hooks consume it. |
| **What may it inspect?** | The completed `Workflow`, including the original query, ordered step arguments/results, canonicalizable trajectory, activated skill IDs, and preliminary execution outcome. |
| **What may it return?** | `True`/`False`, `WorkflowOutcome.SUCCESS`/`FAILURE`, the strings `"success"`/`"failure"`, or `None` to preserve the existing execution outcome. |
| **What happens on `False`?** | The workflow is still persisted for audit, but its outcome is `failure`. It cannot become a successful distillation example and it lowers that trajectory class's promotion success rate. |
| **What happens if the callback raises or returns an unsupported type?** | MemoRizz fails closed for learning by recording `failure`, logs the evaluator problem, and does not break the already-completed user-facing run. |
| **Can it turn an execution failure into success?** | No. A pre-existing execution failure is preserved even if the callback returns success. |
| **Does it approve a refund or prevent a tool call?** | No. Authorization and business invariants belong in application code and transactional tools. The evaluator only decides how the completed trajectory is labeled for learning. |
| **Where does the label matter later?** | Promotion gates use it for `min_success_rate`; distillation selects successful evidence; active-skill monitoring uses it for success/failure and drift; passive shadow evaluation records it as the observed business outcome. |

In this notebook, `workflow_succeeded` deliberately defines success as **the correct business
resolution**, not merely "a refund happened":

1. It rejects missing or malformed tool evidence.
2. It reads the first `lookup_order` result as the authoritative eligibility decision.
3. For a completed order, it requires the exact
   `lookup_order → issue_refund → send_refund_receipt` trajectory and requires every tool result
   to report `ok=True`.
4. For a missing or ineligible order, it requires the agent to stop after `lookup_order`. That
   safe refusal is labeled **success**, because the agent handled the request correctly without
   performing a forbidden side effect.

A production evaluator should be deterministic, fast, side-effect free, and based on structured
tool results or authoritative business state rather than the model's final prose. Keep it
independent of the model being evaluated; an LLM judge here would add latency and could create a
circular definition of success. The callback is runtime-only and is not serialized with a saved
agent, so applications must supply it again when loading the agent.

In [ ]:

def workflow_succeeded(workflow) -> bool:
    raw_steps = list((workflow.steps or {}).values())
    if not raw_steps or not isinstance(raw_steps[0], dict):
        return False
    lookup_result = raw_steps[0].get("result")
    if not isinstance(lookup_result, dict):
        return False
    path = [
        step.get("tool")
        for step in canonical_signature(workflow.steps or {})
    ]
    eligible = (
        lookup_result.get("ok") is True
        and lookup_result.get("status") == "completed"
    )
    expected_path = EXPECTED_REFUND_PATH if eligible else ["lookup_order"]
    if path != expected_path or len(raw_steps) != len(expected_path):
        return False
    if not eligible:
        return (
            lookup_result.get("ok") is False
            or lookup_result.get("status") != "completed"
        )
    return all(
        isinstance(step, dict)
        and isinstance(step.get("result"), dict)
        and step["result"].get("ok") is True
        for step in raw_steps
    )



In [ ]:

def make_agent(
    agent_id: str,
    model,
    instruction: str,
    skill_injection_role: str = "user",
):
    learning_config = dict(
        DEVELOPER_PROMOTION_CONFIG
        if skill_injection_role == "developer"
        else PROMOTION_CONFIG
    )
    agent = MemAgent(
        model=model,
        llm_config=model.get_config(),
        instruction=instruction,
        memory_provider=provider,
        memory_types=[MemoryType.WORKFLOW_MEMORY, MemoryType.SKILLBOX],
        agent_id=agent_id,
        max_steps=8,
        semantic_cache=False,
        continual_learning=True,
        continual_learning_config=learning_config,
        workflow_outcome_evaluator=workflow_succeeded,
        automations_enabled=False,
    )
    for name in list(agent.tool_manager.list_tools()):
        agent.tool_manager.remove_tool(name)
    agent.save()
    return agent

In [ ]:
baseline_agent = make_agent(
    BASELINE_AGENT_ID, baseline_llm, COACHING_INSTRUCTION
)

In [ ]:
user_skill_agent = make_agent(
    USER_SKILL_AGENT_ID, user_skill_llm, COACHING_INSTRUCTION, "user"
)

In [ ]:
developer_skill_agent = make_agent(
    DEVELOPER_SKILL_AGENT_ID,
    developer_skill_llm,
    COACHING_INSTRUCTION,
    "developer",
)

In [ ]:
print(
    {
        "baseline_agent": BASELINE_AGENT_ID,
        "user_skill_agent": USER_SKILL_AGENT_ID,
        "developer_skill_agent": DEVELOPER_SKILL_AGENT_ID,
    }
)

### Toolbox: persisted tool metadata plus live callables

A MemoRizz `Toolbox` is the catalog of functions an agent can use. Registration stores each
tool's name, description, argument schema, ownership, and embedding in the memory provider,
while the current Python process retains the executable callable. Semantic tool search can
discover candidates from the persisted metadata; `initialize_from_toolbox` then attaches the
live functions to each agent's `ToolManager`.

The loop below registers exactly three functions once and loads the same three schemas and
callables into all three arms. It asserts `loaded == 3` so a missing callable or failed Oracle row
cannot silently turn into an apparent model-quality difference.

In [ ]:
from memorizz.long_term.procedural.toolbox import Toolbox

toolbox = Toolbox(
    memory_provider=provider,
    llm_provider=user_skill_llm,
    agent_id=USER_SKILL_AGENT_ID,
)

for tool in (lookup_order, issue_refund, send_refund_receipt):
    toolbox.register_tool(tool)

for agent in (baseline_agent, user_skill_agent, developer_skill_agent):
    loaded = agent.tool_manager.initialize_from_toolbox(toolbox)
    if loaded != 3:
        raise RuntimeError(f"Expected three tools for {agent.agent_id}, loaded {loaded}.")

print("Loaded tools:", user_skill_agent.tool_manager.list_tools())

### Canonical signatures: when different runs count as the same procedure

`canonical_signature(steps)` converts a concrete workflow into ordered units containing the
tool name, sorted argument-key names, an error flag, and a diagnostic retry count. It drops
argument **values**, result payloads, timestamps, and tool database IDs. Consecutive retries
of the same failed tool/argument shape collapse into one unit, and retry count is excluded
from the canonical hash.

Consequently, refund runs for `R-1001` and `R-1002` can share one trajectory class even
though their values and prose differ. Order still matters: a run that omits the receipt or
calls tools in a different sequence gets a different signature. Canonicalization is what
makes evidence counting possible without memorizing customer-specific values.

## 4. Capture Repeated Successful Workflows

All three agents receive the complete procedure while collecting examples. Each request uses a
different order and wording, but a successful run has the same tool sequence.

All arms retain workflows as evidence. During evaluation, the baseline replays its three
uncompressed successful traces in per-request context, while the two treatment agents retrieve
the same reviewed skill at different message roles. This measures workflow-to-skill compression
and isolates role authority without comparing a skill against an unguided model.

The loop stops after each arm has three verified successes, the tutorial's promotion
threshold. Held-out order IDs used later are excluded from these seed runs.

### Seed data and workflow inspection helpers

| Name | Inputs | Detailed behavior |
|---|---|---|
| `SEED_REQUESTS` | `(order_id, query)` pairs | Provides distinct paraphrases and IDs for evidence collection. IDs are values, so they do not affect canonical identity. |
| `agent_workflows(agent_id)` | One agent ID | Reads Oracle `WORKFLOW_MEMORY` and keeps only rows owned by that arm. This explicit filter avoids cross-run evidence leakage. |
| `tool_path(workflow_doc)` | One stored workflow dictionary | Uses the stored canonical signature when present; otherwise rebuilds it from raw steps for backward compatibility. It returns only ordered tool names for grading. |
| `latest_workflow(agent_id, memory_id)` | Agent and run-scoped memory IDs | Finds the workflow produced by one `agent.run` invocation and returns the newest match, or `None` when persistence failed. |

In [ ]:
import uuid
from memorizz.enums.memory_type import MemoryType
from memorizz.long_term.procedural.workflow import canonical_signature


SEED_REQUESTS = [
    ("R-1001", "Please resolve the refund for R-1001."),
    ("R-1002", "Return the payment for completed order R-1002."),
    ("R-1003", "I need my money back for R-1003."),
    ("R-1004", "Process the eligible refund for R-1004."),
    ("R-1005", "Reverse the completed purchase R-1005."),
]


def agent_workflows(agent_id: str):
    rows = provider.list_all(memory_store_type=MemoryType.WORKFLOW_MEMORY) or []
    return [row for row in rows if row.get("agent_id") == agent_id]


def tool_path(workflow_doc: dict):
    signature = workflow_doc.get("canonical_signature")
    if not signature:
        signature = canonical_signature(workflow_doc.get("steps") or {})
    return [step.get("tool") for step in signature]


def latest_workflow(agent_id: str, memory_id: str):
    matches = [
        row
        for row in agent_workflows(agent_id)
        if row.get("memory_id") == memory_id
    ]
    return matches[-1] if matches else None


### Execute, grade, and group the seed runs

For each query, the loop creates independent Oracle snapshots for all three arms, runs each
agent, reloads the stored workflow, and accepts it only when MemoRizz's
business outcome is `success` and its canonical path is exact. Finally it proves that every
accepted treatment run has one canonical hash; otherwise promotion would mix procedures.

| `agent.run` argument | Role in this experiment |
|---|---|
| `query` | The only natural-language task shown to the model. |
| `memory_id=run_id` | Gives workflow and conversation persistence a unique run key and lets the grader retrieve the exact row. |
| `thread_id=uuid4()` | Starts fresh conversational history so earlier answers cannot teach a later trial. |
| `user_id=DEMO_USER_ID` | Exercises tenant-scoped storage with a run-specific demo user. |
| `tool_context={"run_id": run_id}` | Passes a private execution key to Python tools without adding it to their model-visible arguments or return values. |

```mermaid
flowchart TD
    S[Seed query and source order] --> I[Ingest isolated Oracle snapshot]
    I --> A[Run baseline and both skill agents]
    A --> T[Execute live Oracle-backed tools]
    T --> W[(Persist workflow memory)]
    W --> B{Business success and exact path?}
    B -- no --> N[Reject example]
    B -- yes --> H[Collect canonical hash]
    H --> C{Three successes per arm and one shared hash?}
    C -- no --> S
    C -- yes --> P[Ready for promotion]
```

In [ ]:
successful_workflows = {
    "baseline": [],
    "skill_user": [],
    "skill_developer": [],
}

seed_agents = {
    "baseline": baseline_agent,
    "skill_user": user_skill_agent,
    "skill_developer": developer_skill_agent,
}

for order_id, query in SEED_REQUESTS:
    for label, agent in seed_agents.items():
        if len(successful_workflows[label]) >= PROMOTION_CONFIG["min_executions"]:
            continue
        run_id = f"{RUN_ID}-seed-{order_id}-{label}"
        ingest_order_snapshot(run_id, order_id)
        response = agent.run(
            query,
            memory_id=run_id,
            thread_id=str(uuid.uuid4()),
            user_id=DEMO_USER_ID,
            tool_context={"run_id": run_id},
        )

        workflow = latest_workflow(agent.agent_id, run_id)
        path = tool_path(workflow) if workflow else []
        accepted = bool(
            workflow
            and str(workflow.get("outcome", "success")).lower() == "success"
            and path == EXPECTED_REFUND_PATH
        )
        if accepted:
            successful_workflows[label].append(workflow)
        print(
            {
                "arm": label,
                "order_id": order_id,
                "accepted": accepted,
                "path": path,
                "response": response[:100],
            }
        )

    if all(
        len(rows) >= PROMOTION_CONFIG["min_executions"]
        for rows in successful_workflows.values()
    ):
        break

shortfalls = {
    label: len(rows)
    for label, rows in successful_workflows.items()
    if len(rows) < PROMOTION_CONFIG["min_executions"]
}
if shortfalls:
    raise RuntimeError(f"The agents did not produce enough verified seeds: {shortfalls}")

hashes_by_arm = {
    label: {row["canonical_hash"] for row in rows}
    for label, rows in successful_workflows.items()
}
canonical_hashes = {
    next(iter(hashes))
    for hashes in hashes_by_arm.values()
    if len(hashes) == 1
}
if any(len(hashes) != 1 for hashes in hashes_by_arm.values()) or len(canonical_hashes) != 1:
    raise RuntimeError(
        f"Training runs did not form one shared trajectory class: {hashes_by_arm}"
    )

target_hash = next(iter(canonical_hashes))
print(
    {
        "successful_examples": {
            label: len(rows) for label, rows in successful_workflows.items()
        },
        "canonical_hash": target_hash,
    }
)

## 5. Distill, Review, and Activate the Skill

Promotion applies the evidence gates, asks the configured LLM to write a reusable
`SKILL.md`, and stores it in shadow state. 

The small review below checks that all three
tools—including the receipt step—survived distillation before activation.

### Promotion lifecycle, review, and the Oracle retrieval fix

The next block deliberately separates evidence, generation, review, and authority:

1. `promote_class(target_hash)` aggregates only the user-skill agent's matching trajectory
   class, applies the configured gates, samples its real workflows, asks the configured LLM
   to distill a `SKILL.md`, validates the document, and stores the candidate in `shadow`.
2. The notebook reloads that exact `skill_id` and performs an application review: shadow
   status, all three declared tools, an explicit receipt step, and no copied training order
   IDs. Structural library validation and business review are complementary.
3. The reviewed artifact is copied byte-for-byte into the developer-skill agent's Skillbox.
   Only its agent ownership, evidence IDs, and `injection_role` differ. This prevents a second
   stochastic distillation from confounding the message-role comparison.
4. `activate_skill` changes each lifecycle state from `shadow` to `active`; the developer copy
   uses the programmatic `injection_role="developer"` activation option. Only active skills
   participate in semantic retrieval.
5. All agents switch to the same generic production instruction. The baseline has the same
   continual-learning scaffolding and coached workflows but no promoted skill. It receives
   successful workflows as ephemeral context; the treatment arms receive the same skill as
   user or developer instructions. Static system prompts and tool schemas remain identical.


#### Verify that neither skill agent retrieves or replays workflows

`WORKFLOW_MEMORY` remains enabled on both skill agents so each new outcome can be captured for
monitoring and future improvement. That does **not** make raw workflows inference context:
`MemAgent._build_context` retrieves skills separately, while its ordinary pre-inference memory
candidates are knowledge-base and conversation memories—not workflow memory. Prompt assembly
also removes skill-covered workflow items at the final context boundary.

The next cell audits both constructed skill-agent contexts and fails if any retrieved item is
a raw workflow. The benchmark additionally scans every provider prompt: the raw-replay marker
and all seed order IDs must appear in the baseline prompts and in **zero** skill-agent
prompts. The skill arms pass no ephemeral request context; only the baseline receives
`RAW_WORKFLOW_REPLAY`. Capturing each skill arm's new outcome after inference is intentional
and is not workflow retrieval.

In [ ]:
import json
from copy import deepcopy

user_manager = user_skill_agent.continual_learning_manager
developer_manager = developer_skill_agent.continual_learning_manager


### Promote and review the user-authority candidate

The next cell asks MemoRizz to distill the verified trajectory class.

The new skill must
still be in `SHADOW`, declare every required tool, include the receipt step, and avoid
copying any seed order ID. Failing any check stops the notebook before activation.

In [ ]:
from memorizz.long_term.procedural.skillbox import Skill, SkillStatus

promotion_report = user_manager.promote_class(target_hash)

if not promotion_report.promoted:
    raise RuntimeError(f"Skill promotion failed: {promotion_report.to_dict()}")

user_skill_id = promotion_report.promoted[0]
shadow_skill = user_manager.skillbox.get_skill_by_id(user_skill_id)
required_tools = {"lookup_order", "issue_refund", "send_refund_receipt"}

seed_order_ids = {
    order_id
    for order_id, _ in SEED_REQUESTS
    if any(
        order_id in str(row.get("memory_id", ""))
        for row in successful_workflows["skill_user"]
    )
}

leaked_seed_ids = (
    sorted(
        order_id
        for order_id in seed_order_ids
        if order_id in (shadow_skill.content or "")
    )
    if shadow_skill is not None
    else []
)

review_passed = (
    shadow_skill is not None
    and shadow_skill.status == SkillStatus.SHADOW
    and shadow_skill.injection_role.value == "user"
    and required_tools <= set(shadow_skill.tools_used or [])
    and "send_refund_receipt" in (shadow_skill.content or "")
    and not leaked_seed_ids
)

if not review_passed:
    raise RuntimeError(
        "The shadow skill failed business review; "
        f"declared={getattr(shadow_skill, 'tools_used', None)}, "
        f"leaked_seed_ids={leaked_seed_ids}."
    )


### Create the authority-controlled treatment and activate both skills

To isolate instruction authority, the next cell copies the reviewed artifact without
changing its content. One copy remains `user` authority; the other is explicitly approved
as `developer` authority. Activation is a programmatic, reviewed lifecycle transition—not
an automatic result of shadow evaluation.

In [ ]:
# Reuse the exact reviewed artifact so only message authority changes.
developer_rows = successful_workflows["skill_developer"]

developer_shadow_skill = Skill(
    name=shadow_skill.name,
    description=shadow_skill.description,
    content=shadow_skill.content,
    preconditions=list(shadow_skill.preconditions),
    tools_used=list(shadow_skill.tools_used),
    queries=list(shadow_skill.queries),
    agent_id=DEVELOPER_SKILL_AGENT_ID,
    user_id=DEMO_USER_ID,
    source_canonical_hash=target_hash,
    source_workflow_ids=[
        str(row.get("workflow_id") or row.get("_id"))
        for row in developer_rows
    ],
    exemplar_workflow_id=str(
        developer_rows[0].get("_id") or developer_rows[0].get("workflow_id")
    ),
    status=SkillStatus.SHADOW,
    version=shadow_skill.version,
    injection_role="developer",
    baseline=deepcopy(shadow_skill.baseline),
    embedding=deepcopy(shadow_skill.embedding),
)

# Add the skill to the skill box
developer_manager.skillbox.add_skill(developer_shadow_skill)

print(shadow_skill.content)

if not user_manager.activate_skill(user_skill_id):
    raise RuntimeError("The user-authority skill could not be activated.")
if not developer_manager.activate_skill(
    developer_shadow_skill.skill_id,
    injection_role="developer",
):
    raise RuntimeError("The developer-authority skill could not be activated.")


### Hold everything else constant

The following cell switches all arms to the same production instruction, reloads the two
active skills, and verifies that static prompts and tool schemas are identical. It then
builds the baseline's raw-workflow replay payload. After this point, the intended difference
is only procedural representation and message authority.

In [ ]:
for agent in (baseline_agent, user_skill_agent, developer_skill_agent):
    agent.instruction = PRODUCTION_INSTRUCTION

active_user_skill = user_manager.skillbox.get_skill_by_id(user_skill_id)
active_developer_skill = developer_manager.skillbox.get_skill_by_id(
    developer_shadow_skill.skill_id
)
if active_user_skill is None or active_developer_skill is None:
    raise RuntimeError("An activated skill could not be reloaded.")
if (
    active_user_skill.content != active_developer_skill.content
    or active_user_skill.injection_role.value != "user"
    or active_developer_skill.injection_role.value != "developer"
):
    raise RuntimeError("The treatment skills differ by more than authority.")

baseline_skills = (
    baseline_agent.continual_learning_manager.skillbox.list_skills()
)
parity_agents = (baseline_agent, user_skill_agent, developer_skill_agent)
static_prompts_identical = all(
    agent._build_system_prompt() == baseline_agent._build_system_prompt()
    for agent in parity_agents
)
tool_schemas_identical = all(
    agent._build_llm_tools() == baseline_agent._build_llm_tools()
    for agent in parity_agents
)
if baseline_skills or not static_prompts_identical or not tool_schemas_identical:
    raise RuntimeError(
        "The three arms differ outside learned-skill availability/authority: "
        f"baseline_skills={len(baseline_skills)}, "
        f"same_prompt={static_prompts_identical}, "
        f"same_tools={tool_schemas_identical}."
    )

RAW_WORKFLOW_REPLAY = {
    "representation": "uncompressed successful workflow records",
    "guidance": (
        "Treat these past runs as examples, verify the current order, and do not copy "
        "customer-specific values."
    ),
    "successful_workflows": [
        {
            "user_query": row.get("user_query"),
            "outcome": row.get("outcome"),
            "steps": row.get("steps"),
            "canonical_signature": row.get("canonical_signature"),
        }
        for row in successful_workflows["baseline"]
    ],
}

print(
    {
        "user_skill_id": user_skill_id,
        "developer_skill_id": active_developer_skill.skill_id,
        "user_role": active_user_skill.injection_role.value,
        "developer_role": active_developer_skill.injection_role.value,
        "same_skill_content": (
            active_user_skill.content == active_developer_skill.content
        ),
        "static_prompts_identical": static_prompts_identical,
        "tool_schemas_identical": tool_schemas_identical,
        "baseline_skill_count": len(baseline_skills),
    }
)

### Audit retrieval and prompt placement before evaluation

This cell forces Oracle skill-index setup, retrieves each active skill, and inspects the
assembled messages. It must find no workflow-memory item in either skill arm. The user skill
must appear only in user context; the developer skill must appear in a developer message.

In [ ]:
# Fail at the provider boundary if the installed package is stale.
provider._ensure_vector_index(MemoryType.SKILLBOX)

probe_query = "Please refund completed order R-1008 and follow the normal process."

def is_raw_workflow_context(item: dict) -> bool:
    source = str(item.get("source") or item.get("memory_type") or "")
    source = source.strip().lower().replace("-", "_").replace(" ", "_")
    return bool(
        item.get("workflow_id")
        or source in {"workflow", "workflow_memory", "procedural_workflow"}
    )

raw_workflow_context_text = json.dumps(
    RAW_WORKFLOW_REPLAY, ensure_ascii=False, default=str, sort_keys=True
)

skill_audit_specs = {
    "skill_user": (user_skill_agent, user_manager, active_user_skill, "user"),
    "skill_developer": (
        developer_skill_agent,
        developer_manager,
        active_developer_skill,
        "developer",
    ),
}
audit_rows = []

for label, (agent, manager, active_skill, expected_role) in skill_audit_specs.items():
    matches = manager.retrieve_skills_for_query(
        probe_query, user_id=DEMO_USER_ID
    )
    if not matches:
        raise RuntimeError(
            "The active skill was not retrieved for a matching request. "
            f"agent={label}, status={active_skill.status.value}, "
            f"threshold={manager.config.retrieval_min_similarity}. "
            "Confirm the installed package contains the Oracle retrieval fix, then "
            "inspect query/skill similarity."
        )

    audit_context = agent._build_context(
        probe_query,
        memory_id=f"{RUN_ID}-{label}-context-audit",
        user_id=DEMO_USER_ID,
    )
    retrieved_items = list(audit_context.get("retrieved_memories") or [])
    retrieved_workflows = [
        item for item in retrieved_items if is_raw_workflow_context(item)
    ]
    activated_skills = list(audit_context.get("activated_skills") or [])
    prompt_messages = agent._build_prompt_messages(
        agent._build_system_prompt(), probe_query, audit_context
    )
    developer_messages = [
        message for message in prompt_messages if message.get("role") == "developer"
    ]
    final_user_text = prompt_messages[-1].get("content", "")
    role_is_correct = (
        len(developer_messages) == 1
        and active_skill.name in developer_messages[0].get("content", "")
        and active_skill.name not in final_user_text
        if expected_role == "developer"
        else not developer_messages and active_skill.name in final_user_text
    )
    if retrieved_workflows or not activated_skills or not role_is_correct:
        raise RuntimeError(
            "Skill-only context audit failed: "
            f"agent={label}, activated_skills={len(activated_skills)}, "
            f"retrieved_workflows={len(retrieved_workflows)}, "
            f"prompt_roles={[message.get('role') for message in prompt_messages]}."
        )

    skill_context_text = manager.format_skills_prompt_section(
        matches[:1], injection_role=expected_role
    )
    audit_rows.append(
        {
            "Agent": label,
            "Stored authority": matches[0].skill.injection_role.value,
            "Prompt roles": " → ".join(
                message.get("role", "") for message in prompt_messages
            ),
            "Similarity": round(matches[0].similarity, 3),
            "Activated skills": len(activated_skills),
            "Retrieved workflows": len(retrieved_workflows),
            "Skill context chars": len(skill_context_text),
            "Reduction vs raw replay": round(
                1 - len(skill_context_text) / len(raw_workflow_context_text), 3
            ),
        }
    )

### Display the isolation audit

The compact DataFrame below makes the checks easy to scan: stored authority, actual prompt
roles, semantic similarity, activated-skill count, raw-workflow count, and context-size
reduction relative to replaying full workflow records.

In [ ]:
import pandas as pd

skill_context_audit_df = pd.DataFrame(audit_rows)
skill_context_audit_df.style.format(
    {"Similarity": "{:.3f}", "Reduction vs raw replay": "{:.1%}"}
).hide(axis="index").set_caption(
    "Skill authority and raw-workflow isolation audit"
)

## 6. Compare Raw Workflow Replay, User Skills, and Developer Skills

All three agents saw equivalent coached seed requests and have the same generic production
instruction, continual-learning scaffolding, model configuration, and tools. For every
held-out request, the baseline gets its three raw successful workflow records as ephemeral
context; both skill arms get the same reviewed artifact, once in user context and once in a
developer message. Thus every arm receives procedural evidence while the treatments isolate
compression and instruction authority. Each of six
held-out cases uses a fresh thread and independently ingested Oracle snapshot. Arm order is
shuffled with a fixed seed to reduce time-order bias.

The metrics deliberately separate different claims:

| Metric | Definition |
|---|---|
| **Workflow accuracy** | Exact expected tool path **and** correct final Oracle state: refunded plus receipt for eligible orders, unchanged pending state for the guardrail, or no row for a missing order. |
| **Answer accuracy** | A predeclared deterministic text rubric: after normalizing punctuation and common negation contractions, the answer must identify the order and state the correct refund/receipt, pending, or missing outcome without a false success claim. This is transparent and reproducible, but narrower than a semantic human or LLM judge. |
| **Task accuracy** | Both workflow accuracy and answer accuracy pass for the same case. This is the primary quality measure. |
| **Skill authority observed** | Inspects every provider call. The user-skill arm must contain no developer message; the developer-skill arm must contain a developer message carrying the reviewed skill. |
| **Raw workflows in prompt** | Scans every provider call for the raw-replay marker and seed-order IDs. It must be 100% for the replay baseline and 0% for both skill arms. |
| **Prompt/completion/total tokens** | Sum of provider-reported usage over every `generate` call in `MemAgent.run`, including post-tool follow-ups. Cached tokens are shown separately but remain a subset of prompt tokens. Embedding usage and one-time skill distillation are outside this per-request measure. |
| **Provider inference latency** | Wall time spent inside model `generate` calls, including provider network and queueing time. |
| **End-to-end latency** | Wall time around the complete agent run: skill retrieval, embeddings, Oracle reads/writes, orchestration, tools, and all model calls. |

The useful comparison is whether the skill preserves or improves task accuracy while using
less repeated context than raw workflow replay. Styled pandas DataFrames show every held-out
case, the aggregate quality/token/latency metrics, and each treatment-minus-baseline delta. The
output reports raw and skill context character sizes for orientation, then uses
provider-reported tokens for the cost claim.
Latency remains noisy at six cases, so treat it as an observed smoke-test result rather than
a statistically powered or cross-domain production claim.

### Instrument every model call

`InferenceMeter` temporarily wraps the provider's non-streaming `generate` method for one
agent run. It records provider latency and reported token usage, while also checking whether
raw workflow markers or a developer message reached any tool-loop call. The wrapper is
restored immediately after the run.

In [ ]:
import json
import random
import statistics
import time
import uuid
from collections import defaultdict

import pandas as pd
from IPython.display import Markdown, display


class InferenceMeter:
    """Measure every non-streaming provider call made during one agent run."""

    def __init__(self, model, raw_workflow_markers):
        self.model = model
        self.raw_workflow_markers = tuple(
            str(marker).lower() for marker in raw_workflow_markers
        )
        self.events = []
        self._original_generate = None

    def __enter__(self):
        self._original_generate = self.model.generate

        def measured_generate(*args, **kwargs):
            messages = kwargs.get("messages")
            if messages is None and args:
                messages = args[0]
            prompt_text = json.dumps(
                messages or [], ensure_ascii=False, default=str, sort_keys=True
            ).lower()
            raw_workflow_hits = sorted(
                marker for marker in self.raw_workflow_markers if marker in prompt_text
            )
            developer_messages = [
                message
                for message in (messages or [])
                if isinstance(message, dict) and message.get("role") == "developer"
            ]
            started = time.perf_counter()
            result = self._original_generate(*args, **kwargs)
            elapsed = time.perf_counter() - started
            usage = self.model.get_last_usage() or {}
            self.events.append(
                {
                    "seconds": elapsed,
                    "usage": dict(usage),
                    "raw_workflow_hits": raw_workflow_hits,
                    "developer_message_count": len(developer_messages),
                }
            )
            return result

        self.model.generate = measured_generate
        return self

    def __exit__(self, exc_type, exc, traceback):
        self.model.generate = self._original_generate

    @property
    def inference_seconds(self):
        return sum(event["seconds"] for event in self.events)

    @property
    def usage(self):
        totals = defaultdict(int)
        for event in self.events:
            for key, value in event["usage"].items():
                totals[key] += int(value or 0)
        return dict(totals)

    @property
    def usage_available(self):
        token_keys = ("prompt_tokens", "completion_tokens", "total_tokens")
        return any(
            any(event["usage"].get(key) is not None for key in token_keys)
            for event in self.events
        )

    @property
    def raw_workflow_prompt_hits(self):
        return sorted(
            {
                marker
                for event in self.events
                for marker in event["raw_workflow_hits"]
            }
        )

    @property
    def developer_message_rate(self):
        if not self.events:
            return 0.0
        return statistics.mean(
            event["developer_message_count"] > 0 for event in self.events
        )


RAW_WORKFLOW_PROMPT_MARKERS = (
    RAW_WORKFLOW_REPLAY["representation"],
    *(order_id for order_id, _ in SEED_REQUESTS),
)

### Declare the held-out cases and deterministic graders

The next cell fixes the six evaluation requests before any arm runs. It also defines the
answer rubric and the Oracle-state rubric. Eligible cases require the full three-tool path;
pending and missing orders must stop after lookup without claiming a refund.

In [ ]:
EVAL_CASES = [
    {
        "case": "eligible_direct",
        "order_id": "R-1008",
        "query": "Please refund order R-1008 using the normal process.",
        "expected_path": EXPECTED_REFUND_PATH,
        "answer_kind": "refunded",
    },
    {
        "case": "eligible_return_payment",
        "order_id": "R-1009",
        "query": "Return the payment for R-1009 after checking it.",
        "expected_path": EXPECTED_REFUND_PATH,
        "answer_kind": "refunded",
    },
    {
        "case": "eligible_money_back",
        "order_id": "R-1010",
        "query": "The customer needs their money back for R-1010.",
        "expected_path": EXPECTED_REFUND_PATH,
        "answer_kind": "refunded",
    },
    {
        "case": "eligible_reverse_charge",
        "order_id": "R-1011",
        "query": "Reverse the eligible charge for order R-1011.",
        "expected_path": EXPECTED_REFUND_PATH,
        "answer_kind": "refunded",
    },
    {
        "case": "pending_guardrail",
        "order_id": "R-1099",
        "query": "Refund R-1099 only if its current status allows it.",
        "expected_path": ["lookup_order"],
        "answer_kind": "pending",
    },
    {
        "case": "missing_guardrail",
        "order_id": "R-1199",
        "query": "Refund R-1199 if that order exists and is eligible.",
        "expected_path": ["lookup_order"],
        "answer_kind": "missing",
    },
]


def normalize_answer(response: str) -> str:
    text = str(response).lower().replace("’", "'")
    for contraction, expansion in {
        "wasn't": "was not",
        "isn't": "is not",
        "couldn't": "could not",
        "doesn't": "does not",
    }.items():
        text = text.replace(contraction, expansion)
    return " ".join(text.split())


def claims_successful_refund(text: str) -> bool:
    positive = "refunded" in text or (
        "refund" in text
        and any(
            marker in text
            for marker in ("issued", "processed", "completed", "successful", "resolved")
        )
    )
    negated = any(
        phrase in text
        for phrase in (
            "not refunded",
            "no refund",
            "refund was not",
            "refund has not",
            "refund could not",
            "could not refund",
            "unable to refund",
            "cannot refund",
            "not eligible",
        )
    )
    return positive and not negated


def claims_sent_receipt(text: str) -> bool:
    positive = ("receipt" in text or "confirmation" in text) and any(
        marker in text for marker in ("sent", "emailed", "provided")
    )
    negated = any(
        phrase in text
        for phrase in (
            "receipt not sent",
            "receipt was not sent",
            "no receipt",
            "could not send",
            "unable to send",
        )
    )
    return positive and not negated


def answer_is_accurate(case: dict, response: str) -> bool:
    """Transparent lexical rubric fixed before either arm is evaluated."""
    text = normalize_answer(response)
    order_present = case["order_id"].lower() in text
    false_success = claims_successful_refund(text) or claims_sent_receipt(text)
    if case["answer_kind"] == "refunded":
        return (
            order_present
            and claims_successful_refund(text)
            and claims_sent_receipt(text)
        )
    if case["answer_kind"] == "pending":
        pending_present = any(
            phrase in text
            for phrase in ("pending", "not completed", "not eligible", "cannot refund")
        )
        return order_present and pending_present and not false_success
    missing_present = any(
        phrase in text
        for phrase in (
            "not found",
            "could not find",
            "does not exist",
            "order_not_found",
        )
    )
    return order_present and missing_present and not false_success


def database_state_is_accurate(case: dict, snapshot) -> bool:
    if case["answer_kind"] == "refunded":
        return bool(
            snapshot
            and snapshot["status"] == "refunded"
            and snapshot["receipt_sent"]
            and snapshot["lookup_count"] >= 1
        )
    if case["answer_kind"] == "pending":
        return bool(
            snapshot
            and snapshot["status"] == "pending"
            and not snapshot["receipt_sent"]
            and snapshot["lookup_count"] >= 1
        )
    return snapshot is None

### Run and measure each randomized trial

`run_measured` ingests an isolated order snapshot, runs one arm, validates prompt isolation,
reloads the resulting workflow and database state, and returns one metric row. The loop then
shuffles arm order with a fixed seed and executes every arm once per held-out case.

In [ ]:
def run_measured(label: str, agent, case: dict) -> dict:
    run_id = f"{RUN_ID}-eval-{case['case']}-{label}"
    ingest_order_snapshot(run_id, case["order_id"])
    request_context = RAW_WORKFLOW_REPLAY if label == "baseline" else None

    with InferenceMeter(agent.model, RAW_WORKFLOW_PROMPT_MARKERS) as meter:
        started = time.perf_counter()
        response = agent.run(
            case["query"],
            memory_id=run_id,
            thread_id=str(uuid.uuid4()),
            user_id=DEMO_USER_ID,
            context=request_context,
            tool_context={"run_id": run_id},
        )
        end_to_end_seconds = time.perf_counter() - started

    raw_workflow_prompt_hits = meter.raw_workflow_prompt_hits
    expected_raw_workflows = label == "baseline"
    if bool(raw_workflow_prompt_hits) != expected_raw_workflows:
        raise RuntimeError(
            "Raw-workflow prompt isolation failed: "
            f"agent={label}, case={case['case']}, "
            f"hits={raw_workflow_prompt_hits}."
        )
    expected_developer_role = label == "skill_developer"
    developer_role_observed = meter.developer_message_rate > 0
    if developer_role_observed != expected_developer_role:
        raise RuntimeError(
            "Skill-authority prompt isolation failed: "
            f"agent={label}, case={case['case']}, "
            f"developer_message_rate={meter.developer_message_rate:.1%}."
        )

    workflow = latest_workflow(agent.agent_id, run_id)
    path = tool_path(workflow) if workflow else []
    snapshot = order_snapshot(run_id, case["order_id"])
    state_correct = database_state_is_accurate(case, snapshot)
    workflow_correct = path == case["expected_path"] and state_correct
    answer_correct = answer_is_accurate(case, response)
    usage = meter.usage

    return {
        "agent": label,
        "case": case["case"],
        "path": path,
        "database_state": snapshot,
        "workflow_accuracy": workflow_correct,
        "answer_accuracy": answer_correct,
        "task_accuracy": workflow_correct and answer_correct,
        "skill_activated": bool(workflow and workflow.get("skills_activated")),
        "raw_workflows_in_prompt": bool(raw_workflow_prompt_hits),
        "raw_workflow_prompt_hits": raw_workflow_prompt_hits,
        "developer_message_rate": meter.developer_message_rate,
        "llm_calls": len(meter.events),
        "prompt_tokens": usage.get("prompt_tokens", 0),
        "completion_tokens": usage.get("completion_tokens", 0),
        "total_tokens": usage.get(
            "total_tokens",
            usage.get("prompt_tokens", 0) + usage.get("completion_tokens", 0),
        ),
        "cached_tokens": usage.get("cached_tokens", 0),
        "usage_available": meter.usage_available,
        "inference_ms": meter.inference_seconds * 1000,
        "end_to_end_ms": end_to_end_seconds * 1000,
        "response": response,
    }


assert all(
    agent.instruction == baseline_agent.instruction
    for agent in (user_skill_agent, developer_skill_agent)
)
assert static_prompts_identical
assert tool_schemas_identical

agents = {
    "baseline": baseline_agent,
    "skill_user": user_skill_agent,
    "skill_developer": developer_skill_agent,
}
display_labels = {
    "baseline": "Baseline — raw workflows",
    "skill_user": "Skill — user authority",
    "skill_developer": "Skill — developer authority",
}
results = []
arm_rng = random.Random(1707)

for case in EVAL_CASES:
    arm_order = list(agents.items())
    arm_rng.shuffle(arm_order)
    for label, agent in arm_order:
        row = run_measured(label, agent, case)
        results.append(row)

### Inspect every held-out result

Before aggregation, this DataFrame exposes each answer and tool path. It is the quickest place
to diagnose a failed rubric, unexpected developer-message placement, missing skill activation,
or accidental raw-workflow leakage.

In [ ]:
expected_rows = len(EVAL_CASES) * len(agents)
if len(results) != expected_rows:
    raise RuntimeError(f"Expected {expected_rows} evaluation rows, got {len(results)}.")

case_results_df = pd.DataFrame(
    [
        {
            "Agent": display_labels[row["agent"]],
            "Case": row["case"],
            "Tool path": " → ".join(row["path"]),
            "Workflow correct": "✓" if row["workflow_accuracy"] else "✗",
            "Answer correct": "✓" if row["answer_accuracy"] else "✗",
            "Task correct": "✓" if row["task_accuracy"] else "✗",
            "Skill active": "yes" if row["skill_activated"] else "no",
            "Raw workflows in prompt": (
                "yes" if row["raw_workflows_in_prompt"] else "no"
            ),
            "Developer role in prompt": (
                "yes" if row["developer_message_rate"] > 0 else "no"
            ),
            "Total tokens": row["total_tokens"] if row["usage_available"] else None,
            "Inference (ms)": row["inference_ms"],
            "End-to-end (ms)": row["end_to_end_ms"],
            "Response": row["response"],
        }
        for row in results
    ]
).sort_values(["Case", "Agent"], ignore_index=True)

case_results_view = (
    case_results_df.style
    .format(
        {
            "Total tokens": "{:,.0f}",
            "Inference (ms)": "{:,.1f}",
            "End-to-end (ms)": "{:,.1f}",
        },
        na_rep="n/a",
    )
    .set_properties(
        subset=["Response"],
        **{"text-align": "left", "white-space": "normal", "min-width": "280px"},
    )
    .hide(axis="index")
    .set_caption("Held-out evaluation by case")
)
display(case_results_view)

### Aggregate quality, token, and latency metrics

The next cell summarizes each arm independently. Accuracy and prompt-isolation fields are
rates; token fields are means over requests with provider usage data; latency fields show p50
and p95 observations. `Usage coverage` prevents missing token telemetry from looking like zero
cost.

In [ ]:
def percentile(values, probability: float):
    ordered = sorted(values)
    if not ordered:
        return 0.0
    position = (len(ordered) - 1) * probability
    lower = int(position)
    upper = min(lower + 1, len(ordered) - 1)
    weight = position - lower
    return ordered[lower] * (1 - weight) + ordered[upper] * weight


summaries = {}
for label in agents:
    rows = [row for row in results if row["agent"] == label]
    usage_rows = [row for row in rows if row["usage_available"]]
    summaries[label] = {
        "cases": len(rows),
        "workflow_accuracy": statistics.mean(row["workflow_accuracy"] for row in rows),
        "answer_accuracy": statistics.mean(row["answer_accuracy"] for row in rows),
        "task_accuracy": statistics.mean(row["task_accuracy"] for row in rows),
        "skill_activation_rate": statistics.mean(row["skill_activated"] for row in rows),
        "raw_workflow_prompt_rate": statistics.mean(
            row["raw_workflows_in_prompt"] for row in rows
        ),
        "developer_message_rate": statistics.mean(
            row["developer_message_rate"] for row in rows
        ),
        "mean_llm_calls": statistics.mean(row["llm_calls"] for row in rows),
        "mean_prompt_tokens": (
            statistics.mean(row["prompt_tokens"] for row in usage_rows)
            if usage_rows else None
        ),
        "mean_completion_tokens": (
            statistics.mean(row["completion_tokens"] for row in usage_rows)
            if usage_rows else None
        ),
        "mean_total_tokens": (
            statistics.mean(row["total_tokens"] for row in usage_rows)
            if usage_rows else None
        ),
        "mean_cached_tokens": (
            statistics.mean(row["cached_tokens"] for row in usage_rows)
            if usage_rows else None
        ),
        "token_usage_coverage": len(usage_rows) / len(rows),
        "inference_p50_ms": statistics.median(row["inference_ms"] for row in rows),
        "inference_p95_ms": percentile([row["inference_ms"] for row in rows], 0.95),
        "end_to_end_p50_ms": statistics.median(row["end_to_end_ms"] for row in rows),
        "end_to_end_p95_ms": percentile([row["end_to_end_ms"] for row in rows], 0.95),
    }


summary_df = pd.DataFrame(
    [
        {
            "Agent": display_labels[label],
            "Workflow accuracy": summary["workflow_accuracy"],
            "Answer accuracy": summary["answer_accuracy"],
            "Task accuracy": summary["task_accuracy"],
            "Skill activation": summary["skill_activation_rate"],
            "Raw workflows in prompt": summary["raw_workflow_prompt_rate"],
            "Developer role in prompt": summary["developer_message_rate"],
            "Mean prompt tokens": summary["mean_prompt_tokens"],
            "Mean completion tokens": summary["mean_completion_tokens"],
            "Mean total tokens": summary["mean_total_tokens"],
            "Mean cached tokens": summary["mean_cached_tokens"],
            "Mean LLM calls": summary["mean_llm_calls"],
            "Inference p50 (ms)": summary["inference_p50_ms"],
            "Inference p95 (ms)": summary["inference_p95_ms"],
            "End-to-end p50 (ms)": summary["end_to_end_p50_ms"],
            "End-to-end p95 (ms)": summary["end_to_end_p95_ms"],
            "Usage coverage": summary["token_usage_coverage"],
        }
        for label, summary in summaries.items()
    ]
)

percentage_columns = [
    "Workflow accuracy",
    "Answer accuracy",
    "Task accuracy",
    "Skill activation",
    "Raw workflows in prompt",
    "Developer role in prompt",
    "Usage coverage",
]
token_columns = [
    "Mean prompt tokens",
    "Mean completion tokens",
    "Mean total tokens",
    "Mean cached tokens",
]
latency_columns = [
    "Inference p50 (ms)",
    "Inference p95 (ms)",
    "End-to-end p50 (ms)",
    "End-to-end p95 (ms)",
]
summary_formats = {
    **{column: "{:.1%}" for column in percentage_columns},
    **{column: "{:,.0f}" for column in token_columns},
    **{column: "{:,.1f}" for column in latency_columns},
    "Mean LLM calls": "{:.2f}",
}
summary_view = (
    summary_df.style
    .format(summary_formats, na_rep="n/a")
    .hide(axis="index")
    .set_caption("Aggregate quality, context, token, and latency results")
)
display(summary_view)

### Compare each skill treatment with raw workflow replay

Finally, this cell subtracts the baseline from each treatment. Positive accuracy deltas are
better; negative token and latency deltas are lower-cost observations. The generated text is
deliberately conservative: it only claims a benefit when the measured accuracy and token
results support one.

In [ ]:
baseline_summary = summaries["baseline"]
comparison_rows = []
interpretations = []
for treatment_label in ("skill_user", "skill_developer"):
    treatment = summaries[treatment_label]
    token_delta = (
        treatment["mean_total_tokens"] - baseline_summary["mean_total_tokens"]
        if treatment["mean_total_tokens"] is not None
        and baseline_summary["mean_total_tokens"] is not None
        else None
    )
    token_reduction_pct = (
        (baseline_summary["mean_total_tokens"] - treatment["mean_total_tokens"])
        / baseline_summary["mean_total_tokens"]
        * 100
        if token_delta is not None and baseline_summary["mean_total_tokens"]
        else None
    )
    accuracy_delta = (
        treatment["task_accuracy"] - baseline_summary["task_accuracy"]
    ) * 100
    if accuracy_delta < 0:
        interpretation = (
            f"{display_labels[treatment_label]} showed negative transfer; inspect "
            "retrieval, skill content, and guardrails before activation."
        )
    elif token_delta is not None and token_delta < 0:
        interpretation = (
            f"{display_labels[treatment_label]} preserved or improved task accuracy "
            "while using fewer tokens than raw workflow replay in this smoke test."
        )
    elif accuracy_delta > 0:
        interpretation = (
            f"{display_labels[treatment_label]} improved task accuracy; review token "
            "and latency deltas as the measured cost."
        )
    else:
        interpretation = (
            f"{display_labels[treatment_label]} did not improve accuracy or token use "
            "on this run; do not claim a benefit from these cases."
        )
    interpretations.append(interpretation)
    comparison_rows.append(
        {
            "Treatment": display_labels[treatment_label],
            "Workflow accuracy Δ": (
                treatment["workflow_accuracy"]
                - baseline_summary["workflow_accuracy"]
            ) * 100,
            "Answer accuracy Δ": (
                treatment["answer_accuracy"]
                - baseline_summary["answer_accuracy"]
            ) * 100,
            "Task accuracy Δ": accuracy_delta,
            "Mean total-token Δ": token_delta,
            "Token reduction vs raw workflows": token_reduction_pct,
            "Inference p50 Δ (ms)": (
                treatment["inference_p50_ms"]
                - baseline_summary["inference_p50_ms"]
            ),
            "End-to-end p50 Δ (ms)": (
                treatment["end_to_end_p50_ms"]
                - baseline_summary["end_to_end_p50_ms"]
            ),
            "Raw workflows in prompt": treatment["raw_workflow_prompt_rate"],
            "Developer role in prompt": treatment["developer_message_rate"],
        }
    )

comparison_df = pd.DataFrame(comparison_rows)
comparison_view = (
    comparison_df.style
    .format(
        {
            "Workflow accuracy Δ": "{:+.1f} pp",
            "Answer accuracy Δ": "{:+.1f} pp",
            "Task accuracy Δ": "{:+.1f} pp",
            "Mean total-token Δ": "{:+,.0f}",
            "Token reduction vs raw workflows": "{:.1f}%",
            "Inference p50 Δ (ms)": "{:+,.1f}",
            "End-to-end p50 Δ (ms)": "{:+,.1f}",
            "Raw workflows in prompt": "{:.1%}",
            "Developer role in prompt": "{:.1%}",
        },
        na_rep="n/a",
    )
    .hide(axis="index")
    .set_caption("Each learned-skill treatment minus the raw-workflow baseline")
)
display(comparison_view)
display(Markdown("\n".join(f"- {text}" for text in interpretations)))

## 7. Clean Up

The notebook uses unique agent IDs, so cleanup can remove only this run's Toolbox,
Skillbox, workflow, conversation, tool-log, and isolated order-snapshot rows. The shared
Oracle schema, MemoRizz tables, and `MEMORIZZ_GUIDE_ORDERS` table remain in place. Run this
cell manually if an earlier model call fails after the agents were created.

In [ ]:
from collections import defaultdict

from memorizz.enums.memory_type import MemoryType


agent_ids = {
    BASELINE_AGENT_ID,
    USER_SKILL_AGENT_ID,
    DEVELOPER_SKILL_AGENT_ID,
}
id_fields = {
    MemoryType.TOOLBOX: "tool_id",
    MemoryType.SKILLBOX: "skill_id",
    MemoryType.WORKFLOW_MEMORY: "_id",
    MemoryType.CONVERSATION_MEMORY: "_id",
    MemoryType.TOOL_LOG: "_id",
}
deleted = defaultdict(int)

for memory_type, id_field in id_fields.items():
    rows = provider.list_all(memory_store_type=memory_type) or []
    for row in rows:
        if row.get("agent_id") not in agent_ids:
            continue
        record_id = row.get(id_field)
        if record_id and provider.delete_by_id(
            str(record_id), memory_store_type=memory_type
        ):
            deleted[memory_type.value] += 1

for agent_id in agent_ids:
    if provider.delete_memagent(agent_id, cascade=False):
        deleted[MemoryType.MEMAGENT.value] += 1

with provider._get_connection() as connection:
    cursor = connection.cursor()
    try:
        cursor.execute(
            f"DELETE FROM {ORDER_TABLE} WHERE experiment_id = :experiment_id",
            {"experiment_id": RUN_ID},
        )
        deleted["order_snapshots"] = cursor.rowcount
        connection.commit()
    finally:
        cursor.close()

provider.close()
print("Deleted demo rows:", dict(deleted))

## What the Agent Learned

The model itself did not change. MemoRizz stored successful workflows, grouped the common
tool sequence, distilled it into a reviewed skill, and retrieved that skill into the
context of a matching request. That is continual learning in token space.

The three-arm results compare replaying raw successful workflows with retrieving the same
compiled skill at user and developer authority: whether each preserves the exact Oracle-backed
SOP and correct answer, and how many prompt tokens and milliseconds each representation
consumes. Interpret only values produced
by a fresh, error-free run; do not copy the tutorial's thresholds or claim a general effect
from six cases.

For production, use a real ingestion/CDC boundary and least-privilege order repository;
replace the compact grader with business events and outcome metrics; tune retrieval on
labeled queries; include curated failures; require human or automated shadow review; and
monitor attributed post-activation outcomes so a harmful, stale, or mismatched skill can be
demoted.